# 10주차 강의자용 — Cloud Vision 결과 캐시 만들기 (수업 전 1회)

학생이 API 키 없이도 워크북을 진행할 수 있도록, 과일 80장의 Vision 라벨을 **미리 1회 호출해 `vision_cache.json`으로 저장**하고
강의 Drive 데이터 폴더에 올린다. (80회 호출 = 월 무료 1,000건 범위 안)

![API 키 준비](https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/main/week10_cloud_vision/diagrams/11_api_key_setup.png)

| 순서 | 할 일 |
| --- | --- |
| 1 | Google Cloud 콘솔에서 Cloud Vision API 사용 설정 + API 키 만들기 (API 제한: Cloud Vision API) |
| 2 | 이 노트북의 셀을 위에서부터 실행 (키 입력창에 붙여넣기) |
| 3 | 마지막 셀이 Drive `04_데이터_과일80장` 폴더에 `vision_cache.json`을 올리고 gdown으로 내려받기까지 확인 |
| 4 | 수업 후 API 키 삭제 또는 사용 중지 |

> 키는 이 노트북에 저장되지 않는다. 노트북을 공유하기 전에 출력에 키가 보이지 않는지 확인한다.

In [ ]:
!pip -q install gdown

In [ ]:
# 데이터 준비: ① 강의 Google Drive 폴더 → ② GitHub → ③ 직접 업로드 순서로 시도
import csv
import json
import os
import urllib.request
import zipfile
from pathlib import Path

DATA_FOLDER_ID = "1Is7dwskGMYvOxeIRhy0QYMZejj0K8BWE"   # Drive: 10주_강의자료/04_데이터_과일80장
GITHUB_ZIP = (
    "https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/"
    "main/week10_cloud_vision/dataset/week10_fruits.zip"
)
DATA_DIR = Path("week10_fruits")      # 압축을 풀면 생기는 폴더
DRIVE_DIR = Path("week10_drive")      # Drive 폴더를 내려받는 곳
CLASSES = ["apple", "banana", "orange", "lemon"]


def prepare_data():
    """과일 80장 데이터셋을 준비하고, 어디서 받았는지 돌려준다."""
    if (DATA_DIR / "labels.csv").exists():
        return "이미 준비됨"

    zip_path, source = None, None

    # ① Google Drive 폴더 (강의자가 미리 올려 둔 파일, 가장 빠름)
    try:
        import gdown
        gdown.download_folder(id=DATA_FOLDER_ID, output=str(DRIVE_DIR), quiet=True)
        found = list(DRIVE_DIR.rglob("week10_fruits.zip"))
        if found:
            zip_path, source = found[0], "Google Drive"
    except Exception as e:
        print("Drive 다운로드 실패 →", type(e).__name__)

    # ② GitHub (Drive가 막혔을 때)
    if zip_path is None:
        try:
            urllib.request.urlretrieve(GITHUB_ZIP, "week10_fruits.zip")
            zip_path, source = Path("week10_fruits.zip"), "GitHub"
        except Exception as e:
            print("GitHub 다운로드 실패 →", type(e).__name__)

    # ③ 직접 업로드 (왼쪽 파일 창에 week10_fruits.zip 을 끌어다 놓기)
    if zip_path is None and Path("week10_fruits.zip").exists():
        zip_path, source = Path("week10_fruits.zip"), "직접 업로드"

    if zip_path is None:
        raise FileNotFoundError("week10_fruits.zip 을 왼쪽 파일 창에 업로드한 뒤 다시 실행하세요")

    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    return source


source = prepare_data()
rows = list(csv.DictReader(open(DATA_DIR / "labels.csv", encoding="utf-8")))
print("데이터 출처 :", source)
print("이미지 수   :", len(rows))
print("첫 3줄      :", rows[:3])

In [ ]:
# Cloud Vision REST 호출 함수 + API 키 / 캐시 준비
import base64
import requests

VISION_URL = "https://vision.googleapis.com/v1/images:annotate"
GITHUB_CACHE = (
    "https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/"
    "main/week10_cloud_vision/dataset/vision_cache.json"
)


def get_api_key():
    """① Colab 보안 비밀(VISION_API_KEY) → ② 입력창(getpass) 순서로 키를 받는다."""
    if os.environ.get("VISION_API_KEY"):
        return os.environ["VISION_API_KEY"]
    try:
        from google.colab import userdata
        key = userdata.get("VISION_API_KEY")
        if key:
            return key
    except Exception:
        pass
    try:
        from getpass import getpass
        return getpass("Cloud Vision API 키 (없으면 그냥 Enter): ").strip()
    except Exception:
        return ""


def vision_labels(paths, api_key, max_results=10, batch=16):
    """이미지 경로 목록 → {경로: [{description, score}, ...]} (요청 1번에 최대 16장)"""
    result = {}
    for i in range(0, len(paths), batch):
        chunk = paths[i:i + batch]
        requests_body = []
        for p in chunk:
            content = base64.b64encode((DATA_DIR / p).read_bytes()).decode("ascii")
            requests_body.append({
                "image": {"content": content},
                "features": [{"type": "LABEL_DETECTION", "maxResults": max_results}],
            })
        r = requests.post(VISION_URL, params={"key": api_key},
                          json={"requests": requests_body}, timeout=60)
        data = r.json()
        if r.status_code != 200:
            raise RuntimeError(data.get("error", {}).get("message", r.text[:200]))
        for p, resp in zip(chunk, data["responses"]):
            if "error" in resp:
                raise RuntimeError(f"{p}: {resp['error'].get('message')}")
            result[p] = [
                {"description": a["description"], "score": round(a["score"], 4)}
                for a in resp.get("labelAnnotations", [])
            ]
    return result


def load_cache():
    """강의자가 만든 vision_cache.json (Drive 폴더 → GitHub 순서)"""
    found = list(DRIVE_DIR.rglob("vision_cache.json")) + [
        p for p in [Path("vision_cache.json")] if p.exists()]
    if not found:
        try:
            urllib.request.urlretrieve(GITHUB_CACHE, "vision_cache.json")
            found = [Path("vision_cache.json")]
        except Exception:
            return None
    return json.loads(found[0].read_text(encoding="utf-8"))


API_KEY = get_api_key()
CACHE = load_cache()
if API_KEY:
    MODE = "live"
elif CACHE:
    MODE = "cache"
else:
    MODE = "none"

print("Vision 모드:", MODE)
print("  live  = API를 직접 호출 (키 있음)")
print("  cache = 강의자가 미리 저장한 Vision 결과 사용 (키 없음)")
print("  none  = Vision 단계 건너뜀 (분할·kNN 실습은 그대로 가능)")
if CACHE:
    print("캐시 정보:", CACHE["meta"])

assert API_KEY, '강의자 노트북은 API 키가 필요합니다'

In [ ]:
# 80장 전체 호출 (16장씩 5번 요청)
import datetime

paths = [r["path"] for r in rows]
labels = vision_labels(paths, API_KEY, max_results=10)

cache = {
    "meta": {
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "feature": "LABEL_DETECTION",
        "maxResults": 10,
        "images": len(labels),
    },
    "labels": labels,
}
Path("vision_cache.json").write_text(
    json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")
print("저장 완료:", len(labels), "장")
for p in paths[::20]:
    print(p, [(a["description"], a["score"]) for a in labels[p][:4]])

In [ ]:
# Drive 데이터 폴더에 업로드 (이미 있으면 같은 파일 ID로 내용만 교체)
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

auth.authenticate_user()
drive = build("drive", "v3")
query = (f"'{DATA_FOLDER_ID}' in parents and name='vision_cache.json' "
         "and trashed=false")
found = drive.files().list(q=query, fields="files(id,name)").execute()["files"]
media = MediaFileUpload("vision_cache.json", mimetype="application/json")
if found:
    drive.files().update(fileId=found[0]["id"], media_body=media).execute()
    print("기존 파일 교체:", found[0]["id"])
else:
    f = drive.files().create(
        body={"name": "vision_cache.json", "parents": [DATA_FOLDER_ID]},
        media_body=media, fields="id").execute()
    print("새로 업로드:", f["id"])

### (Wokwi 실습용) GitHub에도 올리기

Wokwi 시뮬레이터는 구글 Vision 서버에 HTTPS로 연결하지 못해, Wokwi판 예제 3·4는 GitHub의
`week10_cloud_vision/dataset/vision_cache.json` 을 읽는다. 아래 셀로 파일을 내려받은 뒤
GitHub 저장소 → `week10_cloud_vision/dataset/` → **Add file › Upload files** 로 올린다.
(Colab 노트북의 cache 모드도 Drive가 막히면 이 GitHub 파일을 사용한다.)

In [ ]:
from google.colab import files
files.download("vision_cache.json")   # 브라우저로 내려받기 → GitHub dataset/ 폴더에 업로드

In [ ]:
# 학생 환경과 같은 방법(gdown 폴더 다운로드)으로 받아지는지 확인
import shutil
import gdown

shutil.rmtree("check_drive", ignore_errors=True)
gdown.download_folder(id=DATA_FOLDER_ID, output="check_drive", quiet=True)
print(sorted(p.name for p in Path("check_drive").rglob("*") if p.is_file()))